In [242]:
import pandas as pd 
import numpy as np
import random
import os 
import json
import torch
import pickle
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.utils.data import TensorDataset
from attrdict import AttrDict
from transformers import AdamW, get_linear_schedule_with_warmup
from transformers import BertConfig, BertTokenizer, BertModel

In [260]:
default_path = os.getcwd()
data_path = os.path.join(default_path, '../data')
base_model = os.path.join(default_path, '../base-model')
model_path = os.path.join(default_path, '../models/pseudo')
config_path = os.path.join(default_path, '../config')
log_path = os.path.join(default_path, '../log')
config_file = "bert-base.json"

In [261]:
criteria = 4
seed = 21

In [262]:
data_train = pd.read_csv(os.path.join(data_path, f'untagged_a{criteria}_pseudo_train.csv'))
data_val = pd.read_csv(os.path.join(data_path,f'untagged_a{criteria}_pseudo_val.csv'))
data_test = pd.read_csv(os.path.join(data_path, f'untagged_a{criteria}_pseudo_test.csv'))

In [263]:
data_train.head(3)

,id,subreddit,author,text,type,time,parent_id,o_id,tok_len,label,group_a,group_b,group_c,pseudo
0,448auc,depression,DepressedBones,I finally went to the psychiatrist a few days ...,post,1.454633e+09,448auc,448auc,17,4,True,False,False,4.551276
1,t3_2oue5j,depression,Wikipodiatrist,My diagnosis is mood disorder NOS not otherwis...,comment,1.418216e+09,t1_cmqocpa,cmqpxpa,15,4,True,False,False,5.623147
2,5qs4ul,depression,butmydarkpassenger,I wonder if depression and a messed up sleep s...,post,1.485666e+09,5qs4ul,5qs4ul,29,4,True,False,False,5.361147


In [264]:
data_train['label'] = data_train['pseudo']
data_val['label'] = data_val['pseudo']
data_test['label'] =  data_test['pseudo']

In [265]:
tokenizer = BertTokenizer.from_pretrained(os.path.join(base_model, 'bert-base'), model_max_length=128)
config = BertConfig.from_pretrained(os.path.join(base_model, 'bert-base', 'bert_config.json'))
model = BertModel.from_pretrained(os.path.join(base_model, 'bert-base'), config=config)

In [266]:
with open(os.path.join(config_path, 'training_config.json')) as f:
    training_config = AttrDict(json.load(f))

In [267]:
training_config.device = torch.device("cuda") if torch.cuda.is_available() else "cpu"

In [268]:
training_config

AttrDict({'default_path': '../', 'data_path': 'data', 'log_path': 'log', 'model_path': 'model', 'config_path': 'config', 'seed': 42, 'train_batch_size': 8, 'device': device(type='cuda'), 'eval_batch_size': 8, 'num_epochs': 5, 'gradient_accumulation_steps': 1, 'warmup_proportion': 0, 'adam_epsilon': 1e-08, 'learning_rate': 5e-05, 'do_lower_case': False, 'no_cuda': False, 'max_steps': -1, 'logging_steps': 100})

In [269]:
training_config.learning_rate

5e-05

In [270]:
training_config.model_path = model_path
training_config.log_path = log_path

In [271]:
training_config.seed = seed

In [272]:
training_config.seed

21

In [273]:
class BertDataset(Dataset):
    def __init__(self, data_file):
        self.data = data_file
    
    def __len__(self):
        return len(self.data.label)
    
    def reset_index(self):
        self.data.reset_index(inplace=True, drop=True)
    
    def __getitem__(self, idx):
        '''
        return text, label
        '''
        self.reset_index()
        text = self.data.text[idx]
        label = self.data.label[idx]
        return text, label

In [274]:
class BertProcessor():
    def __init__(self, config, training_config, tokenizer, truncation=True):
        self.tokenizer = tokenizer 
        self.max_len = config.max_position_embeddings
        self.pad = training_config.pad
        self.batch_size = training_config.train_batch_size
        self.truncation = truncation
    
    def convert_data(self, data_file):
        context2 = None    # single sentence classification
        batch_encoding = self.tokenizer.batch_encode_plus(
            [(data_file[idx][0], context2) for idx in range(len(data_file))],   # text, 
            max_length = self.max_len,
            padding = self.pad,
            truncation = self.truncation
        )
        
        features = []
        for i in range(len(data_file)):
            inputs = {k: batch_encoding[k][i] for k in batch_encoding}
            try:
                inputs['label'] = data_file[i][1] 
            except:
                # print('input label 오류')
                inputs['label'] = 0 
            features.append(inputs)
        
        all_input_ids = torch.tensor([f['input_ids'] for f in features], dtype=torch.long)
        all_attention_mask = torch.tensor([f['attention_mask'] for f in features], dtype=torch.long)
        all_token_type_ids = torch.tensor([f['token_type_ids'] for f in features], dtype=torch.long)
        all_labels = torch.tensor([f['label'] for f in features], dtype=torch.long)

        dataset = TensorDataset(all_input_ids, all_attention_mask, all_token_type_ids, all_labels)
        return dataset
    
    def shuffle_data(self, dataset, data_type):
        if data_type == 'train':
            return RandomSampler(dataset)
        elif data_type == 'eval' or data_type == 'test':
            return SequentialSampler(dataset)
        
    def load_data(self, dataset, sampler):
        return DataLoader(dataset, sampler=sampler, batch_size=self.batch_size)

In [275]:
def RMSELoss(yhat,y):
    return torch.sqrt(torch.mean((yhat-y)**2))

In [276]:
class BertRegressor(nn.Module):
    def __init__(self, config, model):
        super(BertRegressor, self).__init__()
        self.model = model
        self.linear = nn.Linear(config.hidden_size, 128)
        self.relu = nn.ReLU()
        self.out = nn.Linear(128, 1)
    
    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        logits = outputs.last_hidden_state[:, 0, :]
        # print(f'logits: {len(logits)}, {len(logits[0])}')
        x = self.linear(logits)
        x = self.relu(x)
        score = self.out(x)
        # print(f'score: {score}')
        return score 

In [277]:
class BertTrainer():
    def __init__(self, config, training_config, model, train_dataloader, eval_dataloader):
        self.config = config
        self.training_config = training_config
        self.model = model
        self.train_dataloader = train_dataloader
        self.eval_dataloader = eval_dataloader
        
    def set_seed(self):
        random.seed(self.training_config.seed)
        np.random.seed(self.training_config.seed)
        torch.manual_seed(self.training_config.seed)
        if not self.training_config.no_cuda and torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.training_config.seed)
    
    def train(self):
        global_step = 0; nb_eval_steps = 0
        train_rmse = []; eval_rmse = []
        t_total = len(self.train_dataloader) // self.training_config.gradient_accumulation_steps * self.training_config.num_epochs

        optimizer = AdamW(self.model.parameters(), lr=self.training_config.learning_rate, eps=self.training_config.adam_epsilon)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(t_total * self.training_config.warmup_proportion), \
                                                    num_training_steps=t_total)
        
        criterion = RMSELoss
        # criterion = nn.MSELoss()
        best_loss = 9999 
        
        self.model.zero_grad()
        for epoch in range(int(self.training_config.num_epochs)):
            train_loss = 0.0; eval_loss = 0.0 
            
            for step, batch in enumerate(self.train_dataloader):
                self.model.train()
                batch = tuple(t.to(self.training_config.device) for t in batch)
                inputs = {
                    "input_ids": batch[0],
                    "attention_mask": batch[1],
                    "token_type_ids": batch[2],
                }
                outputs = self.model(**inputs)
                # print(f'output: {type(outputs)}, {outputs.squeeze}')
                label = batch[3]
                # print(f'label: {label}')
                # print(f'output: {outputs}, {outputs.squeeze()}')
                loss = criterion(outputs.squeeze(), batch[3].type_as(outputs))
                loss.backward()
                
                train_loss += loss.item()
                # torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.training_config.max_grad_norm)
                optimizer.step()
                scheduler.step()
                
                self.model.zero_grad()
            
            print(f'epoch: {epoch + 1} done, train_loss: {train_loss / len(self.train_dataloader)}')
            train_rmse.append(train_loss / len(self.train_dataloader))

            for step2, batch2 in enumerate(self.eval_dataloader):
                self.model.eval()
                batch2 = tuple(t.to(self.training_config.device) for t in batch2)

                with torch.no_grad():
                    inputs = {
                        "input_ids": batch2[0],
                        "attention_mask": batch2[1],
                        "token_type_ids": batch2[2],
                    }
                    label2 = batch2[3]
                    outputs = self.model(**inputs)
                    # print(f'outputs: {outputs.squeeze()}, label: {label2}')
                    tmp_eval_loss = criterion(outputs.squeeze(), label2.type_as(outputs))
                    eval_loss += tmp_eval_loss.mean().item()
                    
                nb_eval_steps += 1

            eval_loss = eval_loss / nb_eval_steps
            eval_rmse.append(eval_loss)
 
        self.save_model(os.path.join(self.training_config.default_path, self.training_config.model_path, f'bert_pseudo_a{criteria}_s{seed}_e5.pt'))
        self.save_log(train_rmse, eval_rmse, epoch+1)   # 3, 21, 42 
        return train_rmse, eval_rmse
            
    def save_log(self, train_mse, eval_mse, epoch):
        with open(os.path.join(self.training_config.default_path, self.training_config.log_path, f'train_{epoch}_mse.pickle'), 'wb') as f:
            pickle.dump(train_mse, f, pickle.HIGHEST_PROTOCOL)  
        
        with open(os.path.join(self.training_config.default_path, self.training_config.log_path, f'eval_{epoch}_mse.pickle'), 'wb') as f:
            pickle.dump(eval_mse, f, pickle.HIGHEST_PROTOCOL)  
    
    def save_model(self, model_name):
        torch.save(self.model.state_dict(), model_name)

In [278]:
training_config.model_path

'C:\\Users\\lamda\\Desktop\\LAMDA_git\\DSM-5\\jupyter\\../models/pseudo'

In [279]:
train_file = BertDataset(data_train)
val_file = BertDataset(data_val)

In [280]:
config.max_position_embeddings = 128
config.max_position_embeddings

128

In [281]:
training_config.pad = 'max_length'
training_config.num_epochs = 5

In [282]:
bert_processor = BertProcessor(config, training_config, tokenizer)

In [283]:
train_dataset = bert_processor.convert_data(train_file)
val_dataset = bert_processor.convert_data(val_file)

C:\Users\lamda\AppData\Local\Temp\ipykernel_22892\1743677856.py:31: DeprecationWarning: an integer is required (got type numpy.float64).  Implicit conversion to integers using __int__ is deprecated, and may be removed in a future version of Python.
  all_labels = torch.tensor([f['label'] for f in features], dtype=torch.long)


In [284]:
train_sampler = bert_processor.shuffle_data(train_dataset, 'train')
val_sampler = bert_processor.shuffle_data(val_dataset, 'eval')

In [285]:
train_dataloader = bert_processor.load_data(train_dataset, train_sampler)
val_dataloader = bert_processor.load_data(val_dataset, val_sampler)

In [286]:
len(train_dataloader), len(val_dataloader)

(346, 39)

In [287]:
model_reg = BertRegressor(config, model).to(training_config.device)  

In [288]:
bert_trainer = BertTrainer(config, training_config, model_reg, train_dataloader, val_dataloader)

In [289]:
data_train.groupby('label').count()

,id,subreddit,author,text,type,time,parent_id,o_id,tok_len,group_a,group_b,group_c,pseudo
label,,,,,,,,,,,,,
0.566276,1,1,1,1,1,1,1,1,1,1,1,1,1
1.032181,1,1,1,1,1,1,1,1,1,1,1,1,1
1.232101,1,1,1,1,1,1,1,1,1,1,1,1,1
1.449308,1,1,1,1,1,1,1,1,1,1,1,1,1
1.453760,1,1,1,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8.989569,1,1,1,1,1,1,1,1,1,1,1,1,1
9.324072,1,1,1,1,1,1,1,1,1,1,1,1,1
9.392579,1,1,1,1,1,1,1,1,1,1,1,1,1


In [290]:
train_mse, eval_mse = bert_trainer.train()

C:\Users\lamda\.conda\envs\aaai\lib\site-packages\transformers\optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


epoch: 1 done, train_loss: 1.040468151341973
epoch: 2 done, train_loss: 0.6027790790779053
epoch: 3 done, train_loss: 0.4124868516890989
epoch: 4 done, train_loss: 0.30238971749395993
epoch: 5 done, train_loss: 0.22202742838359982
